# Aquifer Lens: Groundwater Trend Detective for Saudi Arabia
**NASA Space Apps Challenge 2026 - "Be An Earth System Trend Detective!"**

Is a region gaining or losing water, and is the trend real? This notebook pulls NASA GRACE/GRACE-FO
satellite gravity data (monthly total water storage, 2002 to today), charts the time series for a
region you pick, removes the seasonal cycle, and tests the trend's significance with the
Mann-Kendall test and Sen's slope.

Analysis code and data sources are open. The GRACE download uses a free [NASA Earthdata](https://urs.earthdata.nasa.gov/) account; everything else needs no login.

## Results at a glance (April 2002 - September 2026, 259 GRACE months)
Slopes are gap-aware Sen slopes on deseasonalized anomalies (the real time between months is used, because some months are missing). p-values are Mann-Kendall with the Hamed-Rao correction; they are approximate because the months are not evenly spaced. Confidence intervals are rank-based and not adjusted for autocorrelation.
- Every preset region shows a significant decline in total water storage (Hamed-Rao corrected p from 1.1x10^-14 to 5.9x10^-8):
  - Saudi Arabia (country mask): **-0.62 cm/yr** (Sen, 95% CI -0.64 to -0.60; about -13.3 km^3/yr)
  - Saq/Disi: **-1.18 cm/yr** (CI -1.21 to -1.15)
  - Wadi As-Sirhan: **-1.04 cm/yr** (CI -1.07 to -1.02)
  - Al-Ahsa: **-0.54 cm/yr** (CI -0.55 to -0.52)
- Losses **slowed after 2016**. Saq/Disi: -1.38 cm/yr (2002-2016) vs -0.54 cm/yr (2018-2026); the confidence intervals do not overlap. Saudi Arabia: -0.77 vs -0.54. Wadi As-Sirhan: -1.20 vs -0.53. Al-Ahsa: -0.66 vs -0.47.
- Net Saq/Disi change since 2002: about **-23.7 cm** (seasonally adjusted, last 24 vs first 24 available months).
- Soil moisture (GLDAS, 0-2 m, Saudi Arabia country mask): **+0.09 cm/yr**; derived groundwater proxy (GRACE minus soil moisture): **-0.71 cm/yr** (about -15.2 km^3/yr). The proxy is derived, not measured.
- 10 southwestern-highlands cells show a significant **increase**, surviving a Benjamini-Hochberg multiple-testing correction.

Run all cells to reproduce these numbers from the NASA source data. The GLDAS and Saudi country-mask figures are reproduced in section 7b from the published CSV.

Correction note: an earlier version of this analysis computed slopes by position in the month list, which treated missing months as consecutive and made trends too steep (for example Saudi Arabia GRACE -0.73 cm/yr, now -0.62). All numbers here use real time between months.

In [ ]:
!pip -q install earthaccess xarray netCDF4 pymannkendall matplotlib pandas scipy

## 1. Sign in to NASA Earthdata
The GRACE download needs a free Earthdata Login. Run the cell and follow the prompt.

In [ ]:
import earthaccess
auth = earthaccess.login()  # interactive prompt in Colab

## 2. Download the GRACE/GRACE-FO mascon data (one file, ~45 MB)
JPL RL06.3 v04 mascon grid: monthly terrestrial water storage (groundwater + soil water + surface
water), April 2002 to present, from NASA PO.DAAC.

In [ ]:
import earthaccess
results = earthaccess.search_data(
    short_name="TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4", count=1)
print(results[0]["umm"]["GranuleUR"])
paths = earthaccess.download(results, "./data")
print("downloaded:", paths)

## 3. Pick a region
Presets are places in Saudi Arabia where groundwater matters. Or set your own lat/lon box.

In [ ]:
# Analysis boxes (latitude / longitude). Saudi Arabia is a bounding box here; the published
# country-mask result is reproduced from a CSV in section 7b.
PRESETS = {
    "saudi_arabia_bbox": dict(lat_min=16.0, lat_max=32.5, lon_min=34.5, lon_max=55.7),
    "saq_disi":      dict(lat_min=27.0, lat_max=30.5, lon_min=37.5, lon_max=42.5),
    "al_ahsa":       dict(lat_min=24.5, lat_max=26.0, lon_min=49.0, lon_max=50.7),
    "wadi_as_sirhan": dict(lat_min=28.0, lat_max=31.5, lon_min=39.5, lon_max=44.0),
}
REGION = "saq_disi"  # <- change me
box = PRESETS[REGION]
box


## 4. Regional time series, seasonal removal, significance test

### Gap-aware trend helpers
The next cell defines the slope functions used from here on. They use the actual time between months, so missing months do not inflate a slope.

In [ ]:
# Gap-aware trend helpers. Some months are missing (and there is a gap between GRACE and GRACE-FO),
# so slopes use the real time between observations, not the position in the list.
import numpy as np, pandas as pd
import pymannkendall as mk

def _tyears(s):
    return (s.index.year + (s.index.month - 1) / 12).values

def sen_gap(s):
    """Theil-Sen slope per year over actual time deltas (pandas Series with a datetime index)."""
    s = s.dropna(); t = _tyears(s); v = s.values
    i, j = np.triu_indices(len(v), 1)
    return float(np.median((v[j] - v[i]) / (t[j] - t[i])))

def sen_ci95_gap(s, alpha=1.96):
    """Rank-based 95% CI of the gap-aware Sen slope (equally spaced variance approximation)."""
    s = s.dropna(); t = _tyears(s); v = s.values; n = len(v)
    i, j = np.triu_indices(n, 1)
    sl = np.sort((v[j] - v[i]) / (t[j] - t[i])); N = len(sl)
    C = alpha * np.sqrt(n * (n - 1) * (2 * n + 5) / 18.0)
    return float(sl[int(max(0, (N - C) / 2))]), float(sl[int(min(N - 1, (N + C) / 2))])

def deseason(s):
    """Remove the average for each calendar month."""
    return s - s.groupby(s.index.month).transform("mean")


In [ ]:
import numpy as np, pandas as pd, xarray as xr
import pymannkendall as mk
from scipy import stats
import matplotlib.pyplot as plt

ds = xr.open_dataset(paths[0])
if not np.issubdtype(ds["time"].dtype, np.datetime64):
    t0 = pd.Timestamp("2002-01-01")
    ds = ds.assign_coords(time=("time", pd.to_datetime([t0 + pd.Timedelta(days=float(v)) for v in ds["time"].values])))
sf = "scale_factors" if "scale_factors" in ds else "scale_factor"
ds["lwe_thickness"] = ds["lwe_thickness"] * ds[sf]

sub = ds.sel(lat=slice(box["lat_min"], box["lat_max"]), lon=slice(box["lon_min"], box["lon_max"]))
w = np.cos(np.deg2rad(sub["lat"]))
s = pd.Series(np.asarray(sub["lwe_thickness"].weighted(w).mean(dim=("lat","lon")).values).ravel(),
              index=pd.to_datetime(sub["time"].values)).sort_index()
s = s[~s.index.duplicated(keep="first")]

clim = s.groupby(s.index.month).mean()
anom = s - s.index.month.map(clim)          # deseasonalized
a_ = anom.dropna()
test = mk.original_test(a_.values)           # Mann-Kendall p (classic)
sen_yr = sen_gap(a_)                         # gap-aware Sen slope, cm/year
years = (anom.dropna().index - anom.dropna().index[0]).days / 365.25
sl, ic, r, p, se = stats.linregress(years, anom.dropna().values)

print(f"Region: {REGION}")
print(f"Period: {s.index[0].date()} to {s.index[-1].date()}  ({len(s)} months)")
print(f"Mann-Kendall: {test.trend}, p = {test.p:.2e}")
print(f"Sen's slope:  {sen_yr:.2f} cm water/year (gap-aware)")
print(f"OLS slope:    {sl:.2f} cm water/year (p = {p:.2e})")
print("Significant at 95%:", test.p < 0.05)

## 5. Charts

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                       gridspec_kw={"height_ratios": [1, 1]})
ax[0].plot(s.index, s.values, lw=1, color="tab:blue")
ax[0].set_title(f"GRACE/GRACE-FO total water storage - {REGION} (cm)")
ax[0].set_ylabel("cm water")
ax[0].grid(alpha=.3)
ax[1].plot(anom.index, anom.values, lw=.8, color="tab:gray", alpha=.7)
xx = (anom.dropna().index - anom.dropna().index[0]).days / 365.25
ax[1].plot(anom.dropna().index, sl*xx + ic, color="tab:red", lw=2,
           label=f"trend {sl:.2f} cm/yr (MK p={test.p:.1e})")
ax[1].set_title("Deseasonalized + trend"); ax[1].set_ylabel("cm anomaly")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Trend per grid cell (cm/yr) over the Arabian Peninsula, land only
clim_ds = ds["lwe_thickness"].groupby("time.month").mean("time")
anom_ds = ds["lwe_thickness"].groupby("time.month") - clim_ds
dom = anom_ds.sel(lat=slice(12, 36), lon=slice(32, 60))
tv = (dom["time"].values - dom["time"].values[0]) / np.timedelta64(1, "D") / 365.25
tt = xr.DataArray(tv, dims="time")
slope = ((dom - dom.mean("time")) * (tt - float(tt.mean()))).mean("time") / float(((tt - float(tt.mean()))**2).mean())
mask = ds["land_mask"].sel(lat=slice(12, 36), lon=slice(32, 60))
slope_land = slope.where(mask > 0.5)
fig, ax = plt.subplots(figsize=(9, 7))
slope_land.plot(ax=ax, cmap="RdBu", center=0, robust=True, cbar_kwargs={"label": "cm/yr"})
ax.set_title("Water storage trend per grid cell (deseasonalized), Arabian Peninsula")
plt.show()

## 6. Demonstration example: global temperature (no login needed)
The same trend-significance workflow on NASA GISTEMP, so anyone can learn the method first.

In [ ]:
import io, requests
df = pd.read_csv(io.StringIO(requests.get(
    "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv", timeout=60).text),
    skiprows=1, na_values="***")[["Year", "J-D"]].dropna()
df.columns = ["year", "anom"]
g = pd.Series(df.anom.values, index=pd.to_datetime(df.year.astype(int), format="%Y"))
tg = mk.original_test(g.values)
yy = (g.index - g.index[0]).days / 365.25
sg, ig, *_ = stats.linregress(yy, g.values)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(g.index, g.values, lw=.8, color="tab:gray")
ax.plot(g.index, sg*yy + ig, color="tab:red", lw=2,
        label=f"+{sg*10:.2f} C/decade (MK p={tg.p:.1e})")
ax.set_title("NASA GISTEMP global temperature anomaly"); ax.legend(); ax.grid(alpha=.3)
plt.show()

---
*Built for NASA Space Apps Challenge 2026 with AI coding assistance (declared on the project page).
All analysis runs on real NASA data: GRACE/GRACE-FO (PO.DAAC) and GISTEMP (GISS).*

## 7. Robustness: confidence intervals and autocorrelation check
The standard Mann-Kendall test assumes independent months; GRACE anomalies are autocorrelated,
so we re-run with the Hamed-Rao modified Mann-Kendall test and report 95% confidence intervals
for Sen's slope. The headline trends do not change.

In [ ]:
# Robustness: gap-aware Sen slopes, 95% confidence intervals and autocorrelation-adjusted
# Mann-Kendall p. (The Saudi row is the bounding box; section 7b has the country mask.)
import numpy as np
import pymannkendall as mk

import pandas as pd

def regional_series(ds, lat_min, lat_max, lon_min, lon_max):
    """Area-weighted mean lwe_thickness (cm) over a lat/lon box."""
    sub = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    w = np.cos(np.deg2rad(sub["lat"]))
    s = pd.Series(np.asarray(sub["lwe_thickness"].weighted(w).mean(dim=("lat","lon")).values).ravel(),
                  index=pd.to_datetime(sub["time"].values)).sort_index()
    return s[~s.index.duplicated(keep="first")]

deseasonalize = deseason

print(f"{'region':<16}{'Sen (cm/yr)':>12}{'95% CI':>20}{'MK p':>10}{'mod. MK p':>11}")
rob = {}
for name, bx in PRESETS.items():
    d = deseasonalize(regional_series(ds, **bx)).dropna()
    sen = sen_gap(d)
    lo, hi = sen_ci95_gap(d)
    p_std = mk.original_test(d.values).p
    p_mod = mk.hamed_rao_modification_test(d.values).p  # autocorrelation-adjusted
    rob[name] = dict(sen=round(sen,2), ci=(round(lo,2), round(hi,2)),
                     mk_p=p_std, mk_mod_p=p_mod)
    print(f"{name:<16}{sen:>12.2f}{f'[{lo:.2f}, {hi:.2f}]':>20}{p_std:>10.1e}{p_mod:>11.1e}")

# The southwestern-highlands increases are tested properly in section 9
# (Benjamini-Hochberg correction over every land cell on the map).
print("SW highlands: some grid cells trend upward - section 9 tests which ones survive")
print("a Benjamini-Hochberg multiple-testing correction.")


### 7b. Saudi Arabia (country mask) from the published series
The country-mask series is published as `gldas_grace_saudi_monthly.csv` (raw monthly values). A latitude-longitude box that includes the Red Sea gives a smaller trend (about -0.41 cm/yr), so the country mask is the number to quote for Saudi Arabia.

In [ ]:
# 7b. Saudi Arabia (country mask): GRACE, GLDAS soil moisture and the derived groundwater proxy.
# Reads the published monthly series, so it needs no login. Slopes are gap-aware Sen slopes on
# deseasonalized anomalies; p is Mann-Kendall with the Hamed-Rao correction (approximate, because the
# months are not evenly spaced). The groundwater proxy is derived (GRACE minus GLDAS soil moisture), not measured.
BASE = "https://raw.githubusercontent.com/bassemhamdyhamed/aquifer-lens/main/"
sa = pd.read_csv(BASE + "gldas_grace_saudi_monthly.csv", parse_dates=["month"]).set_index("month")
tws = sa["tws_cm"].dropna(); sm_ = sa["soil_moisture_cm"].dropna()
common = tws.index.intersection(sm_.index)
series = {"GRACE total water storage": deseason(tws),
          "GLDAS soil moisture (0-2 m)": deseason(sm_),
          "Groundwater proxy (derived)": deseason(tws[common]) - deseason(sm_[common])}
AREA_KM2 = 2149690   # Saudi Arabia land area used for the volume estimate (km2)
rows = []
for name, v in series.items():
    lo, hi = sen_ci95_gap(v)
    sen = sen_gap(v)
    rows.append(dict(series=name, months=len(v), sen_cm_per_year=round(sen, 3),
                     ci95=f"[{lo:.3f}, {hi:.3f}]", p_hamed_rao=("<1e-15" if mk.hamed_rao_modification_test(v.values).p < 1e-15 else f"{mk.hamed_rao_modification_test(v.values).p:.1e}"),
                     km3_per_year=round(sen * 1e-5 * AREA_KM2, 1)))
print(pd.DataFrame(rows).to_string(index=False))


## 8. Is the loss speeding up or slowing down?
Piecewise Sen slopes for the Saq/Disi region, split at the 2017-2018 gap between the GRACE and GRACE-FO missions. The net change compares the mean of the last 12 months with the mean of the first 12 on the seasonally-adjusted series.

In [ ]:
# Piecewise Sen slopes for Saq/Disi, split at the 2017-2018 mission gap,
# with a significance test per segment (sen_gap and sen_ci95_gap are defined above).
box_sd = PRESETS["saq_disi"]
sub_sd = ds.sel(lat=slice(box_sd["lat_min"], box_sd["lat_max"]), lon=slice(box_sd["lon_min"], box_sd["lon_max"]))
w_sd = np.cos(np.deg2rad(sub_sd["lat"]))
s_sd = pd.Series(np.asarray(sub_sd["lwe_thickness"].weighted(w_sd).mean(dim=("lat","lon")).values).ravel(),
                 index=pd.to_datetime(sub_sd["time"].values)).sort_index()
s_sd = s_sd[~s_sd.index.duplicated(keep="first")]
clim_sd = s_sd.groupby(s_sd.index.month).mean()
anom_sd = (s_sd - s_sd.index.month.map(clim_sd)).dropna()
anom_sd = anom_sd.groupby(anom_sd.index.to_period("M").to_timestamp()).mean()

for a, b, lab in [("2002-04", "2016-12", "2002-2016"), ("2018-06", "2026-12", "2018-2026")]:
    seg = anom_sd[a:b].dropna()
    sen = sen_gap(seg)
    lo, hi = sen_ci95_gap(seg)
    p = mk.original_test(seg.values).p
    print(f"{lab}: Sen {sen:+.2f} cm/yr, 95% CI [{lo:+.2f}, {hi:+.2f}], Mann-Kendall p = {p:.1e} (n = {len(seg)})")

net = anom_sd.iloc[-24:].mean() - anom_sd.iloc[:24].mean()
print(f"Net change since 2002 (deseasonalized; mean of last 24 available months minus first 24): {net:+.1f} cm")
print("The two confidence intervals do not overlap: the slowdown is statistically significant.")
print("Each segment comes from a single mission, so a GRACE vs GRACE-FO offset cannot explain it.")
print("The split point is set by the 2017-2018 data gap, not detected from the data.")

## 9. Are the map increases real, or just multiple testing?
Testing hundreds of grid cells produces some "significant" cells by chance. A Benjamini-Hochberg correction (false discovery rate 5%) over the land cells of the map region checks which cells survive.

In [ ]:
# Benjamini-Hochberg correction over all land cells of the trend map
lwe = ds["lwe_thickness"].sel(lat=slice(15, 34), lon=slice(33, 57))
mask = ds["land_mask"].sel(lat=slice(15, 34), lon=slice(33, 57)) > 0.5
clim_c = lwe.groupby("time.month").mean("time")
anom_c = (lwe.groupby("time.month") - clim_c)

records = []
for i, la in enumerate(anom_c["lat"].values):
    for j, lo_ in enumerate(anom_c["lon"].values):
        if not mask.values[i, j]:
            continue
        v = anom_c.values[:, i, j]
        v = v[~np.isnan(v)]
        if len(v) < 120:
            continue
        r = mk.original_test(v)
        records.append((float(la), float(lo_), float(r.slope) * 12, float(r.p)))

pvals = np.array([r[3] for r in records])
order = np.argsort(pvals)
ranks = np.empty(len(pvals)); ranks[order] = np.arange(1, len(pvals) + 1)
sig = pvals <= 0.05 * ranks / len(pvals)
sw = [r for k, r in enumerate(records)
      if sig[k] and 16 <= r[0] <= 20 and 42 <= r[1] <= 46 and r[2] > 0]
print(f"Land cells tested: {len(records)}; significant after Benjamini-Hochberg (FDR 5%): {int(sig.sum())}")
print(f"Southwestern-highlands cells (16-20 N, 42-46 E) with a significant INCREASE after correction: {len(sw)}")

## 10. Context layers: rain (IMERG) and greenness (NDVI)
Two context layers for Saudi Arabia and three sub-boxes (rain for Saudi Arabia uses the country mask; the other layers use the Saudi box, 16-32.5 N, 34.5-55.7 E) (Saq/Disi, Qassim, Hail). They show what happened alongside the storage change. This is correlation, not proof of pumping.
- Rain: NASA GPM IMERG Final Precipitation L3 monthly 0.1 deg V07 (GPM_3IMERGM), DOI [10.5067/GPM/IMERG/3B-MONTH/07](https://doi.org/10.5067/GPM/IMERG/3B-MONTH/07). Units: mm/month.
- Greenness: MODIS/Terra Vegetation Indices Monthly L3 Global 0.05 deg CMG V061 (MOD13C2), DOI [10.5067/MODIS/MOD13C2.061](https://doi.org/10.5067/MODIS/MOD13C2.061). NDVI is unitless.

The monthly regional means are published in this repository as `imerg_saudi_monthly.csv` and `ndvi_saudi_monthly.csv`, so this section needs no login and no download from NASA.

In [ ]:
import pandas as pd, numpy as np
import pymannkendall as mk
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/bassemhamdyhamed/aquifer-lens/main/"
LAYERS = {"IMERG rain (mm/month per year)": "imerg_saudi_monthly.csv",
          "MODIS NDVI (per year)": "ndvi_saudi_monthly.csv"}

def layer_trend(series):
    s = series.dropna()
    anom = s - s.groupby(s.index.month).transform("mean")   # remove seasonal cycle
    hr = mk.hamed_rao_modification_test(anom.values)         # autocorrelation-corrected
    ok = mk.original_test(anom.values)                       # classic Mann-Kendall
    return anom, hr.slope * 12, hr.p, ok.p                   # slope per year

rows = []
store = {}
for label, fn in LAYERS.items():
    df = pd.read_csv(BASE + fn, parse_dates=["date"]).set_index("date")
    for area in df.columns:
        anom, slope, p_hr, p_mk = layer_trend(df[area])
        store[(label, area)] = (df[area], anom, slope)
        rows.append(dict(layer=label, area=area, months=int(df[area].notna().sum()),
                         first=df.index[0].strftime("%Y-%m"), last=df[area].dropna().index[-1].strftime("%Y-%m"),
                         sen_per_year=round(slope, 4), p_hamed_rao=round(p_hr, 4), p_classic_mk=round(p_mk, 4)))
pd.DataFrame(rows)

In [ ]:
# Chart: deseasonalized anomalies and the Sen trend line for the Saudi Arabia
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
for a, (label, col) in zip(ax, [("IMERG rain (mm/month per year)", "tab:blue"), ("MODIS NDVI (per year)", "tab:green")]):
    raw, anom, slope = store[(label, "Saudi")]
    t = (anom.index - anom.index[0]).days / 365.25
    ic = np.median(anom.values - slope * t)
    a.plot(anom.index, anom.values, lw=.8, color=col, alpha=.6)
    a.plot(anom.index, ic + slope * t, lw=2.5, color=col)
    a.set_title(f"{label} - Saudi Arabia, deseasonalized anomaly, trend {slope:+.4f}")
    a.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Evapotranspiration (MOD16, yearly)
MODIS/Terra Net Evapotranspiration Gap-Filled Yearly L4 Global 500m V061 (MOD16A3GF), DOI [10.5067/MODIS/MOD16A3GF.061](https://doi.org/10.5067/MODIS/MOD16A3GF.061). Units: mm per year, 2002 to 2025 (24 years), same four boxes. The series is yearly, so there is no seasonal cycle to remove: the test is a plain Mann-Kendall test with Sen's slope. MOD16 is known to overestimate ET over bare desert, so read the level with care; the trend is the more reliable part. Published as `et_saudi_yearly.csv`. The yearly values in the CSV are rounded to 0.1 mm, so the last digit of a slope or p-value can differ slightly from the dashboard, which used unrounded values.

In [ ]:
et = pd.read_csv(BASE + "et_saudi_yearly.csv", index_col="year")
rows = []
for area in et.columns:
    r = mk.original_test(et[area].values)
    rows.append(dict(area=area, years=len(et), first=int(et.index[0]), last=int(et.index[-1]),
                     sen_mm_per_year_per_year=round(r.slope, 2), p_mann_kendall=round(r.p, 3)))
print(pd.DataFrame(rows).to_string(index=False))
ax = et["Saudi"].plot(figsize=(11, 3.5), color="tab:orange", alpha=.7, title="MOD16 ET - Saudi Arabia (mm/yr)")
ax.grid(alpha=.3); plt.show()

### Soil moisture (SMAP, monthly sample)
SMAP L3 Radiometer Global Daily 36 km EASE-Grid Soil Moisture V009 (SPL3SMP), DOI [10.5067/OK8NL42F0B4O](https://doi.org/10.5067/OK8NL42F0B4O). Units: m3/m3, April 2015 to October 2026. Each month is one sample: the 15th-day morning overpass (136 possible samples). SMAP measures roughly the top 5 cm of soil, and retrievals are missing for many months in the sub-boxes (the table shows the sample count). Cells are 36 km wide. Groundwater depletion is deep, so a surface sensor is not expected to show it (an interpretation, not proof). Published as `smap_saudi_monthly.csv`. Missing months are dropped; the Sen slope uses the real time gap between samples, in m3/m3 per year.

In [ ]:
sm = pd.read_csv(BASE + "smap_saudi_monthly.csv", parse_dates=["date"]).set_index("date")
rows = []
for area in sm.columns:
    s = sm[area].dropna()
    anom = s - s.groupby(s.index.month).transform("mean")              # remove seasonal cycle
    t = (anom.index.year + (anom.index.month - 1) / 12).values         # time in years (gaps respected)
    v = anom.values
    sen = np.median([(v[j] - v[i]) / (t[j] - t[i]) for i in range(len(v)) for j in range(i + 1, len(v))])
    p = mk.hamed_rao_modification_test(v).p
    rows.append(dict(area=area, samples=len(s), of=len(sm), sen_m3m3_per_year=round(sen, 5), p_hamed_rao=round(p, 3)))
print(pd.DataFrame(rows).to_string(index=False))
sm["Saudi"].plot(style=".", figsize=(11, 3.5), color="tab:purple", title="SMAP soil moisture - Saudi Arabia (m3/m3)")
plt.grid(alpha=.3); plt.show()

### Cropland cover (MODIS MCD12Q1, yearly)
MODIS/Terra+Aqua Land Cover Type Yearly L3 Global 500m V061 (MCD12Q1), DOI [10.5067/MODIS/MCD12Q1.061](https://doi.org/10.5067/MODIS/MCD12Q1.061). Cropland = IGBP class 12 (croplands) plus class 14 (cropland/natural vegetation mosaic), as a percentage of land pixels in each box, 2002 to 2025. At 500 m, land cover is coarse in an arid mosaic of small fields, and the percentages are small (about 0.1 to 2.3 %), so the direction of change is more reliable than the exact level. Correlation, not proof; changes in cropland cover are consistent with changes in irrigated farming, but this was not tested. Published as `cropland_saudi_yearly.csv` (values rounded to 0.001).

In [ ]:
cr = pd.read_csv(BASE + "cropland_saudi_yearly.csv", index_col="year")
rows = []
for area in cr.columns:
    r = mk.original_test(cr[area].values)
    rows.append(dict(area=area, first=int(cr.index[0]), last=int(cr.index[-1]),
                     pct_first=round(cr[area].iloc[0], 2), pct_last=round(cr[area].iloc[-1], 2),
                     peak_year=int(cr[area].idxmax()), low_year=int(cr[area].idxmin()),
                     sen_pct_per_year=round(r.slope, 4), p_mann_kendall=round(r.p, 3)))
print(pd.DataFrame(rows).to_string(index=False))
cr.plot(figsize=(11, 3.5), title="Cropland cover, % of land (MCD12Q1 classes 12 + 14)")
plt.grid(alpha=.3); plt.show()

Reading the table: a rising rain or NDVI trend does not show where the water comes from. Greening while stored water declines is consistent with irrigation, but this analysis does not prove it.

ET is flat in this record (no significant trend in any box). SMAP shows no significant trend in any box. Cropland cover rose slightly in the Saudi Arabia, fell in Hail and Qassim, and shows no significant overall trend in Saq/Disi.

## Data and satellites

Used for the trend numbers:
- GRACE (NASA/DLR): launched 17 March 2002, science ended October 2017. Twin satellites about 220 km apart, K-band microwave ranging; about 300 km resolution. [NASA fact sheet](https://science.nasa.gov/earth/earth-observatory/grace-fact-sheet/)
- GRACE-FO (NASA/GFZ): launched 22 May 2018, same technique plus an experimental Laser Ranging Interferometer. [NASA JPL](https://gracefo.jpl.nasa.gov/mission/spacecraft/overview/)
- GLDAS Noah (a land-surface model, not a satellite), L4 monthly 0.25 degree V2.1: soil moisture 0-2 m. DOI [10.5067/SXAVCZFAQLNO](https://doi.org/10.5067/SXAVCZFAQLNO)
- GPM Core Observatory, launched 27 February 2014, carries the GMI radiometer and DPR radar. IMERG is a merged product, not a satellite. Used for the rain layer. [NASA GPM](https://gpm.nasa.gov/missions/GPM/core-observatory)
- MODIS on Terra (18 December 1999) and Aqua (4 May 2002): 36 bands, 2,330 km swath. Used for the NDVI layer. [NASA MODIS](https://modis.gsfc.nasa.gov/about/design.php)

Related, not used in the numbers:
- SMAP, launched 31 January 2015, L-band radiometer for soil moisture (its radar failed in July 2015). [NASA JPL](https://www.jpl.nasa.gov/missions/soil-moisture-active-passive-smap/)

Note: the groundwater proxy (GRACE total storage minus GLDAS soil moisture) is derived, not measured.